In [37]:
import numpy as np
import pandas as pd

In [38]:
# Loading data and separate features/targets

path = '..\Dataset\simulation_matrix_updated_info.csv'

data = pd.read_csv(path)

data["Max_Equiv_Stress_Pa"] = data["Max_Equiv_Stress_Pa"].astype(str).str.replace(" ", "").astype(float)

print(f"Loaded dataset: {data.shape[0]} rows, {data.shape[1]} columns")
data.head()

Loaded dataset: 81 rows, 8 columns


,ID,Blade_Length_m,Root_Chord_mm,Material,Applied_Load_Pa,Status,Max_Deformation_m,Max_Equiv_Stress_Pa
0,BL0.8_C150_AL_P500,0.8,150,Aluminium,500,Updated,0.002438,4299500.0
1,BL0.8_C150_AL_P1000,0.8,150,Aluminium,1000,Updated,0.004875,8599100.0
2,BL0.8_C150_AL_P1500,0.8,150,Aluminium,1500,Updated,0.007313,12899000.0
3,BL0.8_C150_FBG_P500,0.8,150,Fiberglass,500,Updated,0.006742,4304200.0
4,BL0.8_C150_FBG_P1000,0.8,150,Fiberglass,1000,Updated,0.013483,8608500.0


In [39]:
#Defining features and targets
FEATURE_COLS = ["Blade_Length_m", "Root_Chord_mm", "Material", "Applied_Load_Pa"]
TARGET_STRESS = "Max_Equiv_Stress_Pa"
TARGET_DEFLECTION = "Max_Deformation_m"

X = data[FEATURE_COLS].copy()
y_stress = data[TARGET_STRESS].copy()
y_deflection = data[TARGET_DEFLECTION].copy()

print("Features (X):", X.shape)
print("Target 1 - Max Stress:", y_stress.shape)
print("Target 2 - Max Deflection:", y_deflection.shape)
X.head()

Features (X): (81, 4)
Target 1 - Max Stress: (81,)
Target 2 - Max Deflection: (81,)


,Blade_Length_m,Root_Chord_mm,Material,Applied_Load_Pa
0,0.8,150,Aluminium,500
1,0.8,150,Aluminium,1000
2,0.8,150,Aluminium,1500
3,0.8,150,Fiberglass,500
4,0.8,150,Fiberglass,1000


In [40]:
# One hot encoding material feature
X_encoded = pd.get_dummies(X, columns=["Material"], prefix="Material")
print("Columns after encoding:", X_encoded.columns.tolist())
X_encoded.head()

Columns after encoding: ['Blade_Length_m', 'Root_Chord_mm', 'Applied_Load_Pa', 'Material_Aluminium', 'Material_Carbon fiber', 'Material_Fiberglass']


,Blade_Length_m,Root_Chord_mm,Applied_Load_Pa,Material_Aluminium,Material_Carbon fiber,Material_Fiberglass
0,0.8,150,500,True,False,False
1,0.8,150,1000,True,False,False
2,0.8,150,1500,True,False,False
3,0.8,150,500,False,False,True
4,0.8,150,1000,False,False,True


In [41]:
# Train-test split (80/20, seprately per target)

from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

X_train_stress, X_test_stress, y_train_stress, y_test_stress = train_test_split(
    X_encoded, y_stress, test_size=0.2, random_state=RANDOM_STATE
)

X_train_deflection, X_test_deflection, y_train_deflection, y_test_deflection = train_test_split(
    X_encoded, y_deflection, test_size=0.2, random_state=RANDOM_STATE
)

print("Stress model     -> Train:", X_train_stress.shape, " Test:", X_test_stress.shape)
print("Deflection model -> Train:", X_train_deflection.shape, " Test:", X_test_deflection.shape)
print("\nFeature names:", X_encoded.columns.tolist())

Stress model     -> Train: (64, 6)  Test: (17, 6)
Deflection model -> Train: (64, 6)  Test: (17, 6)

Feature names: ['Blade_Length_m', 'Root_Chord_mm', 'Applied_Load_Pa', 'Material_Aluminium', 'Material_Carbon fiber', 'Material_Fiberglass']


In [42]:
# Scaling on numeric features
from sklearn.preprocessing import StandardScaler

NUMERIC_COLS = ["Blade_Length_m", "Root_Chord_mm", "Applied_Load_Pa"]

def scale_split(X_train_raw, X_test_raw, numeric_cols):
    scaler = StandardScaler()
    X_train_scaled = X_train_raw.copy()
    X_test_scaled = X_test_raw.copy()
    X_train_scaled[numeric_cols] = scaler.fit_transform(X_train_raw[numeric_cols])
    X_test_scaled[numeric_cols] = scaler.transform(X_test_raw[numeric_cols])
    return X_train_scaled, X_test_scaled, scaler

X_train_stress_scaled, X_test_stress_scaled, scaler_stress = scale_split(
    X_train_stress, X_test_stress, NUMERIC_COLS
)
X_train_deflection_scaled, X_test_deflection_scaled, scaler_deflection = scale_split(
    X_train_deflection, X_test_deflection, NUMERIC_COLS
)

X_train_stress_scaled.head()

,Blade_Length_m,Root_Chord_mm,Applied_Load_Pa,Material_Aluminium,Material_Carbon fiber,Material_Fiberglass
61,1.169187,-1.105542,-0.073922,False,True,False
55,1.169187,-1.105542,-0.073922,True,False,False
40,-0.037716,-0.301511,-0.073922,False,False,True
9,-1.244619,-0.301511,-1.256676,True,False,False
64,1.169187,-0.301511,-0.073922,True,False,False


In [43]:
# final validation
def validate(name, X_train, X_test, y_train, y_test, feature_names):
    print(f"--- {name} ---")
    print("No missing values (train):", X_train.isnull().sum().sum() == 0)
    print("No missing values (test):", X_test.isnull().sum().sum() == 0)
    print("X_train shape:", X_train.shape, " X_test shape:", X_test.shape)
    print("y_train shape:", y_train.shape, " y_test shape:", y_test.shape)
    print("Feature names:", feature_names)
    print()

validate("Stress model", X_train_stress_scaled, X_test_stress_scaled,
          y_train_stress, y_test_stress, X_train_stress_scaled.columns.tolist())
validate("Deflection model", X_train_deflection_scaled, X_test_deflection_scaled,
          y_train_deflection, y_test_deflection, X_train_deflection_scaled.columns.tolist())

--- Stress model ---
No missing values (train): True
No missing values (test): True
X_train shape: (64, 6)  X_test shape: (17, 6)
y_train shape: (64,)  y_test shape: (17,)
Feature names: ['Blade_Length_m', 'Root_Chord_mm', 'Applied_Load_Pa', 'Material_Aluminium', 'Material_Carbon fiber', 'Material_Fiberglass']

--- Deflection model ---
No missing values (train): True
No missing values (test): True
X_train shape: (64, 6)  X_test shape: (17, 6)
y_train shape: (64,)  y_test shape: (17,)
Feature names: ['Blade_Length_m', 'Root_Chord_mm', 'Applied_Load_Pa', 'Material_Aluminium', 'Material_Carbon fiber', 'Material_Fiberglass']



In [44]:
# making a sub-folder to store all train and test datasets
import os
os.makedirs("ml_ready_data", exist_ok=True)

X_train_stress_scaled.to_csv("ml_ready_data/X_train_stress.csv", index=False)
X_test_stress_scaled.to_csv("ml_ready_data/X_test_stress.csv", index=False)
y_train_stress.to_csv("ml_ready_data/y_train_stress.csv", index=False)
y_test_stress.to_csv("ml_ready_data/y_test_stress.csv", index=False)

X_train_deflection_scaled.to_csv("ml_ready_data/X_train_deflection.csv", index=False)
X_test_deflection_scaled.to_csv("ml_ready_data/X_test_deflection.csv", index=False)
y_train_deflection.to_csv("ml_ready_data/y_train_deflection.csv", index=False)
y_test_deflection.to_csv("ml_ready_data/y_test_deflection.csv", index=False)

print("Saved ML-ready datasets to ./ml_ready_data/")

Saved ML-ready datasets to ./ml_ready_data/
